# trial-047 — standalone EBM vs standalone LightGBM

Два фиксированных stratified folds, seed 2026, четыре фичи trial-043, без target encoding. Notebook сам находит competition `train.csv`, считает OOF ROC AUC, fold-wise дельты, bootstrap и время. После Run All скачайте `trial-047-output.zip` из секции Output.

In [ ]:
%pip install -q interpret-core==0.7.8 lightgbm==4.7.0

In [ ]:
from __future__ import annotations

import json
import platform
import time
import zipfile
from importlib.metadata import version
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from interpret.glassbox import ExplainableBoostingClassifier
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

RUN_ID = 'trial-047'
TARGET = 'addicted_label'
ID_COLUMN = 'id'
FOLDS = 2
SEED = 2026
BOOTSTRAP_REPEATS = 100
BOOTSTRAP_SAMPLE_SIZE = 296_302
MISSING_SENTINEL = -999.0
EXACT_LGBM_REFERENCE = 0.9637208761621594
EXACT_LGBM_FOLD_REFERENCE = [0.9638540806419047, 0.9635876703258506]
RAW_NUMERIC = [
    'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
    'work_study_hours', 'sleep_hours', 'notifications_per_day',
    'app_opens_per_day', 'weekend_screen_time',
]
FEATURES = [
    *RAW_NUMERIC, 'academic_work_impact', 'other_screen_time_hours',
    'screen_social_load', 'high_stress_other_screen_per_sleep',
    'work_study_screen_share', 'entertainment_screen_share',
    'notifications_per_day_value_frequency',
    'daily_screen_time_hours_value_frequency',
]
REQUIRED_COLUMNS = {ID_COLUMN, TARGET, 'stress_level', *RAW_NUMERIC, 'academic_work_impact'}
OUTPUT_DIR = Path('/kaggle/working')

def find_train_csv() -> Path:
    candidates = list(Path('/kaggle/input').rglob('train.csv'))
    for path in candidates:
        try:
            columns = set(pd.read_csv(path, nrows=2).columns)
        except Exception:
            continue
        if REQUIRED_COLUMNS.issubset(columns):
            return path
    raise FileNotFoundError(
        'Competition train.csv not found. Attach the competition data to this notebook.'
    )

def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator / denominator.where(denominator > 0)

def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result['other_screen_time_hours'] = (
        result['daily_screen_time_hours'] - result['social_media_hours']
        - result['gaming_hours'] - result['work_study_hours']
    )
    result['screen_social_load'] = (
        result['daily_screen_time_hours'] * result['social_media_hours']
    )
    high_stress = result['stress_level'].map({'High': 1.0, 'Medium': 0.0, 'Low': 0.0})
    valid_sleep = result['sleep_hours'].where(result['sleep_hours'] > 0)
    result['high_stress_other_screen_per_sleep'] = (
        high_stress * result['other_screen_time_hours'] / valid_sleep
    )
    result['work_study_screen_share'] = safe_ratio(
        result['work_study_hours'], result['daily_screen_time_hours']
    )
    result['entertainment_screen_share'] = safe_ratio(
        result['social_media_hours'] + result['gaming_hours'],
        result['daily_screen_time_hours'],
    )
    result['notifications_per_day_value_frequency'] = 0.0
    result['daily_screen_time_hours_value_frequency'] = 0.0
    assert not any('target_encoding' in column for column in result.columns)
    return result

def fold_frequency(source: pd.Series, train_idx: np.ndarray, valid_idx: np.ndarray):
    train_keys = source.iloc[train_idx].astype('string').fillna('__MISSING__')
    valid_keys = source.iloc[valid_idx].astype('string').fillna('__MISSING__')
    mapping = train_keys.value_counts(dropna=False) / len(train_idx)
    return (
        train_keys.map(mapping).to_numpy(dtype=np.float32),
        valid_keys.map(mapping).fillna(0.0).to_numpy(dtype=np.float32),
    )

def numeric_matrix(frame: pd.DataFrame) -> np.ndarray:
    result = frame[FEATURES].copy()
    result['academic_work_impact'] = result['academic_work_impact'].map({'No': 0.0, 'Yes': 1.0})
    result = result.replace([np.inf, -np.inf], np.nan).fillna(MISSING_SENTINEL)
    values = result.to_numpy(dtype=np.float32)
    if not np.isfinite(values).all():
        raise ValueError('Non-finite model matrix')
    return values

def percentile_rank(values: np.ndarray) -> np.ndarray:
    return pd.Series(np.asarray(values).reshape(-1)).rank(method='average', pct=True).to_numpy()

def bootstrap_deltas(y, control, candidate):
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    n_pos = int(round(BOOTSTRAP_SAMPLE_SIZE * len(pos) / len(y)))
    n_neg = BOOTSTRAP_SAMPLE_SIZE - n_pos
    sampled_y = np.r_[np.ones(n_pos, dtype=np.int8), np.zeros(n_neg, dtype=np.int8)]
    rng = np.random.default_rng(SEED)
    deltas = []
    for _ in range(BOOTSTRAP_REPEATS):
        idx = np.r_[rng.choice(pos, n_pos, replace=True), rng.choice(neg, n_neg, replace=True)]
        deltas.append(roc_auc_score(sampled_y, candidate[idx]) - roc_auc_score(sampled_y, control[idx]))
    values = np.asarray(deltas)
    return {
        'repeats': BOOTSTRAP_REPEATS, 'sample_size': BOOTSTRAP_SAMPLE_SIZE,
        'mean_auc_delta': float(values.mean()),
        'ci95_lower': float(np.quantile(values, 0.025)),
        'ci95_upper': float(np.quantile(values, 0.975)),
        'probability_delta_positive': float(np.mean(values > 0)),
        'scope': 'row-sampling uncertainty only',
    }

started = time.perf_counter()
train_path = find_train_csv()
print(f'Train: {train_path}', flush=True)
train = build_features(pd.read_csv(train_path))
y = train[TARGET].to_numpy(dtype=np.int8)
row_count = len(train)
lgb_probability = np.full(row_count, np.nan)
ebm_probability = np.full(row_count, np.nan)
lgb_rank = np.full(row_count, np.nan)
ebm_rank = np.full(row_count, np.nan)
fold_number = np.zeros(row_count, dtype=np.int8)
fold_results = []
importance_rows = []
splitter = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, valid_idx) in enumerate(splitter.split(np.zeros(row_count), y), 1):
    fold_started = time.perf_counter()
    train_fold = train.iloc[train_idx].copy()
    valid_fold = train.iloc[valid_idx].copy()
    for feature, source in [
        ('notifications_per_day_value_frequency', 'notifications_per_day'),
        ('daily_screen_time_hours_value_frequency', 'daily_screen_time_hours'),
    ]:
        train_values, valid_values = fold_frequency(train[source], train_idx, valid_idx)
        train_fold[feature] = train_values
        valid_fold[feature] = valid_values
    x_train = numeric_matrix(train_fold)
    x_valid = numeric_matrix(valid_fold)
    y_train, y_valid = y[train_idx], y[valid_idx]
    print(f'Fold {fold}/{FOLDS}: LightGBM start', flush=True)
    lgb_model = lgb.LGBMClassifier(
        objective='binary', n_estimators=200, learning_rate=0.10,
        num_leaves=31, max_depth=-1, min_child_samples=20,
        reg_alpha=0.0, reg_lambda=0.0, random_state=SEED, n_jobs=-1, verbosity=-1,
    )
    fit_started = time.perf_counter()
    lgb_model.fit(x_train, y_train)
    lgb_seconds = time.perf_counter() - fit_started
    fold_lgb_probability = lgb_model.predict_proba(x_valid)[:, 1]
    print(f'Fold {fold}/{FOLDS}: EBM start', flush=True)
    ebm_model = ExplainableBoostingClassifier(
        feature_names=FEATURES, feature_types=['continuous'] * len(FEATURES),
        max_bins=1024, max_interaction_bins=64, interactions='3x',
        validation_size=0.15, outer_bags=14, learning_rate=0.015,
        max_rounds=50000, early_stopping_rounds=100, max_leaves=2,
        n_jobs=-2, random_state=SEED,
    )
    fit_started = time.perf_counter()
    ebm_model.fit(x_train, y_train)
    ebm_seconds = time.perf_counter() - fit_started
    fold_ebm_probability = ebm_model.predict_proba(x_valid)[:, 1]
    fold_lgb_rank = percentile_rank(fold_lgb_probability)
    fold_ebm_rank = percentile_rank(fold_ebm_probability)
    lgb_probability[valid_idx] = fold_lgb_probability
    ebm_probability[valid_idx] = fold_ebm_probability
    lgb_rank[valid_idx] = fold_lgb_rank
    ebm_rank[valid_idx] = fold_ebm_rank
    fold_number[valid_idx] = fold
    lgb_auc = roc_auc_score(y_valid, fold_lgb_rank)
    ebm_auc = roc_auc_score(y_valid, fold_ebm_rank)
    fold_results.append({
        'fold': fold, 'lightgbm_auc': lgb_auc, 'ebm_auc': ebm_auc,
        'delta_vs_notebook_lightgbm': ebm_auc - lgb_auc,
        'delta_vs_project_exact_lightgbm': ebm_auc - EXACT_LGBM_FOLD_REFERENCE[fold - 1],
        'lightgbm_fit_seconds': lgb_seconds, 'ebm_fit_seconds': ebm_seconds,
        'elapsed_seconds': time.perf_counter() - fold_started,
    })
    for term, importance in zip(ebm_model.term_names_, ebm_model.term_importances()):
        importance_rows.append({'fold': fold, 'term': term, 'importance': float(importance)})
    print(f'Fold {fold}: LGBM={lgb_auc:.10f}, EBM={ebm_auc:.10f}, delta={ebm_auc-lgb_auc:+.10f}, EBM time={ebm_seconds:.1f}s', flush=True)

if not all(np.isfinite(values).all() for values in [lgb_rank, ebm_rank]):
    raise ValueError('Missing OOF predictions')
lgb_auc = roc_auc_score(y, lgb_rank)
ebm_auc = roc_auc_score(y, ebm_rank)
bootstrap = bootstrap_deltas(y, lgb_rank, ebm_rank)
metrics = {
    'run_id': RUN_ID, 'model': 'ExplainableBoostingClassifier',
    'feature_source': 'trial-043 four-feature bundle; no target encoding',
    'rows': row_count, 'folds': FOLDS, 'seed': SEED, 'features': FEATURES,
    'preprocessing': 'shared numeric matrix; missing=-999; academic No=0 Yes=1 missing=-999; frequency fold-safe',
    'versions': {'python': platform.python_version(), 'interpret-core': version('interpret-core'), 'lightgbm': lgb.__version__},
    'parameters': {'outer_bags': 14, 'interactions': '3x', 'max_rounds': 50000, 'learning_rate': 0.015, 'early_stopping_rounds': 100},
    'oof': {
        'lightgbm_auc': lgb_auc, 'ebm_auc': ebm_auc,
        'delta_vs_notebook_lightgbm': ebm_auc - lgb_auc,
        'project_exact_lightgbm_reference_auc': EXACT_LGBM_REFERENCE,
        'delta_vs_project_exact_lightgbm': ebm_auc - EXACT_LGBM_REFERENCE,
        'spearman_vs_notebook_lightgbm': float(spearmanr(ebm_rank, lgb_rank).statistic),
    },
    'fold_results': fold_results, 'paired_bootstrap_vs_notebook_lightgbm': bootstrap,
    'elapsed_seconds': time.perf_counter() - started,
}
metrics_path = OUTPUT_DIR / f'metrics_{RUN_ID}_ebm.json'
oof_path = OUTPUT_DIR / f'oof_predictions_{RUN_ID}.csv'
importance_path = OUTPUT_DIR / f'term_importance_{RUN_ID}.csv'
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame({
    'row_index': np.arange(row_count), 'id': train[ID_COLUMN], 'target': y,
    'fold': fold_number, 'lightgbm_probability': lgb_probability,
    'lightgbm_fold_rank': lgb_rank, 'ebm_probability': ebm_probability,
    'ebm_fold_rank': ebm_rank,
}).to_csv(oof_path, index=False)
importance = pd.DataFrame(importance_rows).groupby('term', as_index=False)['importance'].mean().sort_values('importance', ascending=False)
importance.to_csv(importance_path, index=False)
zip_path = OUTPUT_DIR / f'{RUN_ID}-output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in [metrics_path, oof_path, importance_path]:
        archive.write(path, arcname=path.name)
print(json.dumps(metrics['oof'], indent=2), flush=True)
print(f'Ready: {zip_path}', flush=True)